In [ ]:
import os
os.environ["WANDB_START_METHOD"] = "thread"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import wandb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv(r"C:\Users\17063\OneDrive\Documents\GitHub\DS_Capstone_Group_1\data\v2_cleaned\all_merged.csv")

FEATURES = [
    'uninsured_pct', 'high_bp_pct', 'asthma_pct', 'no_checkup_pct',
    'no_dental_visit_pct', 'diabetes_pct', 'high_cholesterol_pct',
    'median_household_income', 'poverty_rate_pct', 'total_population_poverty',
    'md_primary_care_count', 'do_primary_care_count',
    'total_population', 'pcp_count', 'pcp_per_100k'
]
TARGET = 'hpsa_score_max'

# Training data: HPSA-designated counties only
train_df = df[df['hpsa_designated'] == 1].dropna(subset=FEATURES + [TARGET])

# Imputation targets: non-designated counties
impute_df = df[df['hpsa_designated'] == 0].dropna(subset=FEATURES).copy()

X = train_df[FEATURES]
y = train_df[TARGET]
groups = train_df['geoid']

# Split by county (geoid) to avoid leaking county-level scores into the test set
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Training tracts : {len(X_train):,}  ({groups.iloc[train_idx].nunique()} counties)")
print(f"Test tracts     : {len(X_test):,}  ({groups.iloc[test_idx].nunique()} counties)")
print(f"Imputation rows : {len(impute_df):,}")

In [ ]:
# Skewness check
skewness = train_df[FEATURES + [TARGET]].skew().sort_values(key=abs, ascending=False)
print("Skewness (sorted by absolute value):")
print(skewness.round(3).to_string())
print("\nHighly skewed (|skew| > 1):")
print(skewness[skewness.abs() > 1].round(3).to_string())

# Histograms for all features + target
n_cols = 4
n_rows = -(-len(FEATURES + [TARGET]) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(FEATURES + [TARGET]):
    axes[i].hist(train_df[col].dropna(), bins=40, edgecolor='none')
    axes[i].set_title(f"{col}\nskew={train_df[col].skew():.2f}", fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

corr = train_df[FEATURES + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=ax, annot_kws={"size": 7})
ax.set_title("Feature Correlation Heatmap", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of hpsa_score_max for designated counties (hpsa_designated == 1)
scores = train_df[TARGET].dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(scores, bins=40, edgecolor='none')
axes[0].set_title("hpsa_score_max distribution (designated counties)")
axes[0].set_xlabel("hpsa_score_max")
axes[0].set_ylabel("count")

axes[1].boxplot(scores, vert=True)
axes[1].set_title("hpsa_score_max boxplot")
axes[1].set_ylabel("hpsa_score_max")

plt.tight_layout()
plt.show()

print(scores.describe().round(2))
print(f"\nSkewness: {scores.skew():.3f}")

In [ ]:
def evaluate_model(name, pipeline, config=None):
    """Fit pipeline, evaluate on test set, log metrics to WandB."""
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    metrics = {
        "mae":  mean_absolute_error(y_test, preds),
        "rmse": mean_squared_error(y_test, preds) ** 0.5,
        "r2":   r2_score(y_test, preds),
    }
    if USE_WANDB:
        run = wandb.init(
            project="hpsa-score-imputation",
            name=name,
            config=config or {},
            reinit=True
        )
        wandb.log(metrics)
        run.finish()
    print(f"{name:<25} MAE={metrics['mae']:.3f}  RMSE={metrics['rmse']:.3f}  R\u00b2={metrics['r2']:.3f}")
    return pipeline

In [ ]:
USE_WANDB = True  # set to False to skip WandB logging

In [ ]:
trained_models = {}

# 1. Linear Regression
trained_models['linear_regression'] = evaluate_model(
    'linear_regression',
    Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    config={'model_type': 'LinearRegression'}
)

# 2. Decision Tree
trained_models['decision_tree'] = evaluate_model(
    'decision_tree',
    DecisionTreeRegressor(random_state=42),
    config={'model_type': 'DecisionTreeRegressor', 'random_state': 42}
)

# 3. Random Forest
trained_models['random_forest'] = evaluate_model(
    'random_forest',
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    config={'model_type': 'RandomForestRegressor', 'n_estimators': 100, 'random_state': 42}
)

# 4. Gradient Boosting
trained_models['gradient_boosting'] = evaluate_model(
    'gradient_boosting',
    GradientBoostingRegressor(n_estimators=100, random_state=42),
    config={'model_type': 'GradientBoostingRegressor', 'n_estimators': 100, 'random_state': 42}
)

In [ ]:
# Feature importance for tree-based models
for name in ['random_forest', 'gradient_boosting']:
    model = trained_models[name]
    estimator = model.named_steps['model'] if hasattr(model, 'named_steps') else model
    importances = pd.Series(estimator.feature_importances_, index=FEATURES)
    print(f"\n{name} — top features:")
    print(importances.sort_values(ascending=False).round(4).to_string())

In [ ]:
# Impute hpsa_score_max for hpsa_designated == 0 using all models
X_impute = impute_df[FEATURES]

for name, pipeline in trained_models.items():
    impute_df[f'hpsa_score_max_pred_{name}'] = pipeline.predict(X_impute)

pred_cols = [c for c in impute_df.columns if c.startswith('hpsa_score_max_pred')]
print(impute_df[['geoid_tract', 'geoid'] + pred_cols].head(10))
print(f"\nNull predictions: {impute_df[pred_cols].isnull().sum().sum()}")